In [65]:
# Importations
import os
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import round,col, dayofmonth,month,year,quarter,when 
from pyspark.sql import functions as F

In [66]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [67]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

In [68]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerSilverTransformation") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

26/05/09 01:34:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [69]:
# Récupérer les tables windturbinepower_cleaned et weatherforecastapi_clean du schéma silver dans des DataFrames Spark

#  Récupération des données de la table silver.windpowerturbine_cleaned
windturbinepower_cleaned_df = spark.read.jdbc(
    url=JDBC_URL,
    table="silver.windpowerturbine_cleaned",
    properties=JDBC_PROPS
)

# Récupération des données de la table silver.weatherforecastapi_cleaned
weatherforecastapi_cleaned_df = spark.read.jdbc(
    url=JDBC_URL,
    table="silver.weatherforecastapi_cleaned",
    properties=JDBC_PROPS
)

In [70]:
# Préparation des colonnes et jointure pour construire la table de synthèse
windturbinepower_selected_df = windturbinepower_cleaned_df.select(
    "production_id",
    "date",
    F.date_format(col("time"), "HH:mm:ss").alias("time"),
    "latitude",
    "longitude",
    "region",
    "turbine_name",
    "capacity",
    "status",
    "responsible_department",
    "wind_speed",
    "wind_direction",
    "energy_produced",
    "day",
    "month",
    "quarter",
    "year",
    "hour_of_day",
    "minute_of_hour",
    "second_of_minute",
    "time_period"
)

weatherapi_selected_df = weatherforecastapi_cleaned_df.select(
    "weather_id",
    "date",
    F.date_format(col("time"), "HH:mm:ss").alias("time"),
    "region",
    "region_name",
    "wind_gusts_10m",
    "temperature_2m",
    "cloud_cover"
)

enriched_df = (
    windturbinepower_selected_df.alias("w")
    .join(
        weatherapi_selected_df.alias("m"),
        on=["date", "time", "region"],
        how="left"
    )
    .select(
        col("w.production_id").alias("production_id"),
        col("m.weather_id").alias("weather_id"),
        col("w.date").alias("date"),
        col("w.time").alias("time"),
        col("w.latitude").alias("latitude"),
        col("w.longitude").alias("longitude"),
        col("w.region").alias("region"),
        col("m.region_name").alias("region_name"),
        col("w.turbine_name").alias("turbine_name"),
        col("w.capacity").alias("capacity"),
        col("w.status").alias("status"),
        col("w.responsible_department").alias("responsible_department"),
        col("w.energy_produced").alias("energy_produced"),
        col("w.wind_speed").alias("wind_speed"),
        col("m.wind_gusts_10m").alias("wind_gust_10m"),
        col("w.wind_direction").alias("wind_direction"),
        col("m.temperature_2m").alias("temperature_2m"),
        col("m.cloud_cover").alias("cloud_cover"),
        col("w.day").alias("day"),
        col("w.month").alias("month"),
        col("w.quarter").alias("quarter"),
        col("w.year").alias("year"),
        col("w.hour_of_day").alias("hour_of_day"),
        col("w.minute_of_hour").alias("minute_of_hour"),
        col("w.second_of_minute").alias("second_of_minute"),
        col("w.time_period").alias("time_period")
    )
)
enriched_df = enriched_df.orderBy("production_id","date","time","region")

enriched_df.show(5)

+-------------+----------+----------+--------+---------+-----------+--------+--------------------+------------+--------+----------+----------------------+---------------+----------+-------------+--------------+--------------+-----------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|production_id|weather_id|      date|    time| latitude|  longitude|  region|         region_name|turbine_name|capacity|    status|responsible_department|energy_produced|wind_speed|wind_gust_10m|wind_direction|temperature_2m|cloud_cover|day|month|quarter|year|hour_of_day|minute_of_hour|second_of_minute|time_period|
+-------------+----------+----------+--------+---------+-----------+--------+--------------------+------------+--------+----------+----------------------+---------------+----------+-------------+--------------+--------------+-----------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|         6049|         1|2024-06-15|00:00:00|34.

In [71]:
# Création de la table de synthèse silver.windturbinepower_enriched
create_table_sql = """
CREATE TABLE IF NOT EXISTS silver.windturbinepower_enriched (
    prod_enriched_id BIGSERIAL PRIMARY KEY,
    production_id INTEGER,
    weather_id BIGINT,
    date DATE,
    time TIME,
    latitude NUMERIC(9,6),
    longitude NUMERIC(9,6),
    region VARCHAR(100),
    region_name VARCHAR(255),
    turbine_name VARCHAR(100),
    capacity INTEGER,
    status VARCHAR(50),
    responsible_department VARCHAR(100),
    energy_produced NUMERIC(12,2),
    wind_speed NUMERIC(6,2),
    wind_gust_10m NUMERIC(6,2),
    wind_direction VARCHAR(50),
    temperature_2m NUMERIC(5,2),
    cloud_cover NUMERIC(5,2),
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER,
    hour_of_day INTEGER,
    minute_of_hour INTEGER,
    second_of_minute INTEGER,
    time_period VARCHAR(20)
);
"""

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
        cur.execute(create_table_sql)

        # Contrainte UNIQUE nécessaire pour ON CONFLICT (production_id, date, time)
        try:
            cur.execute("""
                ALTER TABLE silver.windturbinepower_enriched
                ADD CONSTRAINT unique_windturbinepower_enriched_business_key
                UNIQUE (production_id, date, time);
            """)
            print("Contrainte UNIQUE ajoutée sur (production_id, date, time)")
        except (psycopg.errors.DuplicateObject, psycopg.errors.DuplicateTable):
            print("Contrainte UNIQUE existe déjà")

        print("Table silver.windturbinepower_enriched prête")

Contrainte UNIQUE ajoutée sur (production_id, date, time)
Table silver.windturbinepower_enriched prête


In [72]:
# Insertion des données dans la table en mode UPSERT
total_count = enriched_df.count()
print(f"Lignes enrichies à traiter: {total_count}")

enriched_df_pd = enriched_df.toPandas()

upsert_sql = """
INSERT INTO silver.windturbinepower_enriched (
    production_id,
    weather_id,
    date,
    time,
    latitude,
    longitude,
    region,
    region_name,
    turbine_name,
    capacity,
    status,
    responsible_department,
    energy_produced,
    wind_speed,
    wind_gust_10m,
    wind_direction,
    temperature_2m,
    cloud_cover,
    day,
    month,
    quarter,
    year,
    hour_of_day,
    minute_of_hour,
    second_of_minute,
    time_period
)
VALUES (%(production_id)s, %(weather_id)s, %(date)s, %(time)s, %(latitude)s, %(longitude)s, %(region)s, %(region_name)s, %(turbine_name)s, %(capacity)s, %(status)s, %(responsible_department)s, %(energy_produced)s, %(wind_speed)s, %(wind_gust_10m)s, %(wind_direction)s, %(temperature_2m)s, %(cloud_cover)s, %(day)s, %(month)s, %(quarter)s, %(year)s, %(hour_of_day)s, %(minute_of_hour)s, %(second_of_minute)s, %(time_period)s)
ON CONFLICT ON CONSTRAINT unique_windturbinepower_enriched_business_key DO UPDATE SET
    weather_id = EXCLUDED.weather_id,
    latitude = EXCLUDED.latitude,
    longitude = EXCLUDED.longitude,
    region = EXCLUDED.region,
    region_name = EXCLUDED.region_name,
    turbine_name = EXCLUDED.turbine_name,
    capacity = EXCLUDED.capacity,
    status = EXCLUDED.status,
    responsible_department = EXCLUDED.responsible_department,
    energy_produced = EXCLUDED.energy_produced,
    wind_speed = EXCLUDED.wind_speed,
    wind_gust_10m = EXCLUDED.wind_gust_10m,
    wind_direction = EXCLUDED.wind_direction,
    temperature_2m = EXCLUDED.temperature_2m,
    cloud_cover = EXCLUDED.cloud_cover,
    day = EXCLUDED.day,
    month = EXCLUDED.month,
    quarter = EXCLUDED.quarter,
    year = EXCLUDED.year,
    hour_of_day = EXCLUDED.hour_of_day,
    minute_of_hour = EXCLUDED.minute_of_hour,
    second_of_minute = EXCLUDED.second_of_minute,
    time_period = EXCLUDED.time_period
RETURNING xmax = 0 AS inserted;
"""

new_rows_count: int = 0
duplicates_avoided_count: int = 0

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        for _, row in enriched_df_pd.iterrows():
            params = {str(k): v for k, v in row.items()}
            cur.execute(upsert_sql, params)
            result = cur.fetchone()
            inserted = bool(result[0]) if result is not None else False
            if inserted:
                new_rows_count += 1
            else:
                duplicates_avoided_count += 1

print(f"{total_count} lignes traitées avec UPSERT")
print(f"Nouvelles lignes: {new_rows_count}")
print(f"Doublons évités: {duplicates_avoided_count}")

Lignes enrichies à traiter: 432
432 lignes traitées avec UPSERT
Nouvelles lignes: 432
Doublons évités: 0


In [73]:
# Arrêt de la session Spark
spark.stop()